# IFCA Server

> The core abstraction for `IFCA` server.

In [ ]:
#| default_exp servers.ifca

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os

import copy
from tqdm import tqdm
from loguru import logger

import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.client_selector import BaseClientSelector
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

from fedai.servers.base_server import BaseServer
from fedai.utils.registery import AlgorithmRegistry
from fedai.core import get_clean_kwargs
import gc

<torch._C.Generator>

In [ ]:
#| export
@AlgorithmRegistry.register_server("ifca")
class ServerIFCA(BaseServer):
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        super().__init__(cfg, client_selector, client_cls, criterion, fds, writer, device, **kwargs) 
        self.k_models = [copy.deepcopy(self.model) for _ in range(self.cfg.algorithm.K)]

In [ ]:
#| export
@patch
def client_fn(self: ServerIFCA, id, comm_round, client_state):
    if (comm_round == 1 and client_state == {}) or client_state == {}:
        client_state['k_models'] = [self.k_models[i].state_dict() for i in range(self.cfg.algorithm.K)]

    models = [create_model(self.cfg) for _ in range(self.cfg.algorithm.K)]
    for i in range(self.cfg.algorithm.K):
        models[i].load_state_dict(client_state['k_models'][i])

    client_state['k_models'] = models

    kwargs = get_clean_kwargs(self.cfg.optimizer)
    kwargs.pop("cls", None)
    client_state['optimizer'] = [get_optimizer, kwargs]

    train_loader = prepare_dl(self.cfg.data, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg.data, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    client.logger = self.logger

    return client


### Evaluation

In [ ]:
#| export
@patch
def evaluate(self: ServerIFCA, t):
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):
        if id not in self.lst_active_ids:
            continue
        client_state = self.state_mgr.get_state(id)
        client = self.client_fn(id= id, comm_round= t, client_state= client_state)
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
        
        del client
        gc.collect()
        torch.cuda.empty_cache()

    return lst_train_res, lst_test_res    


### Aggregation


In [ ]:
#| export
@patch
def aggregate(self: ServerIFCA, lst_active_ids, comm_round, len_clients_ds):
    m_t = sum(len_clients_ds.values())
    self.lst_active_ids = lst_active_ids
    
    with torch.no_grad():

        for cluster_id in range(self.cfg.algorithm.K):
            global_model = None
            for id in lst_active_ids:
                if self.state_mgr.get_state(id).get('cluster_id', None) != cluster_id:
                    continue
                self.logger.info(f"Aggregating client {id} for cluster {cluster_id}")
                client_state_dict = self.state_mgr.get_state(id).get('model', None)

                if global_model is None:
                    global_model = {k: torch.zeros_like(v) for k, v in client_state_dict.items()}

                n_k = len_clients_ds[id]
                weight = n_k / m_t
                for key in global_model.keys():
                    global_model[key].add_(client_state_dict[key], alpha=weight)

            self.k_models[cluster_id].load_state_dict(global_model) if global_model is not None else self.k_models[cluster_id] 

        for id in lst_active_ids:
            self.state_mgr.set_state(id, {
                    'k_models': [copy.deepcopy(self.k_models[i].state_dict()) for i in range(self.cfg.algorithm.K)],
                    'optimizer': self.state_mgr.get_state(id).get('optimizer', None),
                    'cluster_id': self.state_mgr.get_state(id).get('cluster_id', None),
                    'one_hot_cluster_id': self.state_mgr.get_state(id).get('one_hot_cluster_id', None)
            })
        

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()